**DISCLAIMER**

This repository contains adversarial prompts and sensitive text used solely to evaluate the safety boundaries of
Large Language Models. Content is provided for academic and red-teaming purposes only, does not reflect the views of
the authors, and may be offensive or distressing. Proceed with discretion.

In [1]:
import os
os.environ["UNSLOTH_FIXED_ROPE"] = "1"

In [2]:
from unsloth import FastLanguageModel
import torch
import warnings
import gc
from pathlib import Path

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
from parameters import Parameters
from templates import Templates
from queries import Queries
from generator import generate_prompt

In [4]:
warnings.filterwarnings("ignore", category=FutureWarning, module="transformers.modeling_attn_mask_utils")

In [5]:
# Models
max_seq_length = Parameters.MAX_SEQ_LENGTH
path_to_models = Parameters.PATH_TO_MODELS

model_configurations = [
    {"name": "baseline", "path": str(path_to_models / Parameters.MODEL_NAME_BASELINE)},
    {"name": "abliterated", "path": str(path_to_models / Parameters.MODEL_NAME_ABLITERATED)},
    {"name": "jailbrek_pre_tar", "path": str(path_to_models / Parameters.MODEL_NAME_JAILBREAK_PRE_TAR)},
    {"name": "tar", "path": str(Parameters.MODEL_NAME_TAR)},
    {"name": "jailbreak_post_tar", "path": str(path_to_models / Parameters.MODEL_NAME_JAILBREAK_POST_TAR)},
]

In [6]:
def execute_inference(model, tokenizer, query, system_prompt, prefill):

    if not tokenizer.chat_template or 'start_header_id' not in tokenizer.chat_template:
        tokenizer.chat_template = Templates.LLAMA3_CHAT_TEMPLATE

    prompt = generate_prompt(
        tokenizer=tokenizer,
        system_prompt=system_prompt,
        query=query,
        prefill=prefill,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    prompt_length = inputs.input_ids.shape[1]
    max_new_tokens = max(1, 1024 - prompt_length)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        min_new_tokens=128,
        max_length=None,
        use_cache=True,
        temperature=0.1,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=[tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|eot_id|>")]
    )

    # Decode only the new tokens
    new_tokens = outputs[0][prompt_length:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return answer

In [7]:
# Parameters
model_nickname = "baseline"  # ["baseline", "abliterated", jailbrek_pre_tar", "tar", "jailbreak_post_tar"]
query = Queries.HARMFUL # [Queries.HARMLESS, Queries.HARMFUL, Queries.FALSE_POSITIVE]
prefill = Templates.PREFILL  # ["", Templates.PREFILL]
system_prompt = Templates.SYSTEM_PROMPT_JAILBREAK  # [Templates.SYSTEM_PROMPT_JAILBREAK, Templates.SYSTEM_PROMPT_BASELINE]

In [ ]:
path = next((item["path"] for item in model_configurations if item["name"] == model_nickname), None)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = path,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    device_map = {"": 0},
)

model = FastLanguageModel.for_inference(model)
model.eval()

answer = execute_inference(model, tokenizer, query, system_prompt, prefill)

print(f"\n[🤖 MODEL]: {model_nickname} ({path})")
print(f"\n[⚙️ SYSTEM PROMPT]: {system_prompt}")
print(f"\n[💬 PREFILL]: {prefill}")
print(f"\n[❓ QUERY]: {query}")
print(f"\n[💡 ANSWER]: {answer}")

# Cleanup VRAM
del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()

==((====))==  Unsloth 2025.10.4: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.559 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
